# CSI Data Collector — v2
**Wi-Fi Sensing with ESP32-S3 | Final Course Project — UFMG**

This version generates, in addition to the raw `.csv` data file, a `.json` metadata file per session.  
Intended usage flow:

1. **Cell 1 — Imports & utilities** *(run once per kernel session)*
2. **Cell 2 — Fixed setup metadata** *(fill in once per collection day)*
3. **Cell 3 — Session parameters** *(fill in before each individual session)*
4. **Cell 4 — Data collection** *(run to start recording)*
5. **Cell 5 — Wrap-up & JSON export** *(run immediately after the recording stops)*


---
## Cell 1 — Imports & utilities
Run once at the beginning of the work session.

In [3]:
import serial
import time
import json
import os
from datetime import datetime, timezone
from pathlib import Path

# ──────────────────────────────────────────────
# Hardware constants (unchanged between sessions)
# ──────────────────────────────────────────────
PORT  = "/dev/ttyUSB0"   # serial port of the RX node
BAUD  = 921600           # must match the csi_recv firmware baud rate

HEADER = (
    "timestamp_host,"
    "type,id,mac,rssi,rate,sig_mode,mcs,bandwidth,smoothing,"
    "not_sounding,aggregation,stbc,fec_coding,sgi,noise_floor,"
    "ampdu_cnt,channel,secondary_channel,local_timestamp,ant,"
    "sig_len,rx_state,len,first_word,data\n"
)

# Locate repo root regardless of where Jupyter was launched from
_p = Path.cwd()
for PROJECT_ROOT in [_p, _p.parent, _p.parent.parent]:
    if (PROJECT_ROOT / 'requirements.txt').exists():
        break

FOLDER_NAME = "main"  # change for each collection campaign
DATA_DIR = PROJECT_ROOT / 'data' / FOLDER_NAME
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────
# Utility functions
# ──────────────────────────────────────────────
def _ts_iso() -> str:
    """Returns a local ISO 8601 timestamp with UTC offset."""
    return datetime.now().astimezone().isoformat(timespec='seconds')

def _ts_host() -> str:
    """Host timestamp used to prefix each CSV row."""
    return datetime.now().strftime("%Y%m%dT%H%M%S.%f")

def _build_filename(session_id: str, label: str, dt: datetime) -> str:
    return f"session_{session_id}_{label}_{dt.strftime('%Y%m%d_%H%M')}"

print("Imports OK — Cell 1 ready.")

---
## Cell 2 — Fixed setup metadata
Fill in **once per collection day** (or whenever the physical setup changes).  
These values will be embedded in every JSON file generated during this work session.

In [ ]:
#  FILL IN MANUALLY

SETUP_META = {

    # ── Physical environment ───────────────────────────────────────
    "room": {
        "description"       : "Residential bedroom, brick walls, concrete slab ceiling",
        "dimensions_m"      : {"east_west": 3.40, "north_south": 3.45, "ceiling_height": 2.85},
        "door_material"     : "plywood",
        "window_material"   : "iron and glass",
        "notable_objects"   : ["MDF wardrobe", "mirror", "bed", "desk", "book niches x2"]
    },

    # ── Node positioning ───────────────────────────────────────────
    "nodes": {
        "tx": {
            "role"                     : "STA (ICMP transmitter)",
            "device"                   : "ESP32-S3-DevKitC-1",
            "tripod_height_m"          : 1.20,
            "pos_x_from_west_wall_m"   : 1.45,
            "pos_y_from_north_wall_m"  : 3.08,
            "mac"                      : "1a:00:00:00:00:00"   # <-- fill in
        },
        "rx": {
            "role"                     : "AP (CSI receiver)",
            "device"                   : "ESP32-S3-DevKitC-1",
            "tripod_height_m"          : 1.20,
            "pos_x_from_west_wall_m"   : 1.45,
            "pos_y_from_north_wall_m"  : 1.14,
            "mac"                      : "1c:db:d4:9d:93:dc",  # <-- fill in
            "serial_port"              : PORT,
            "baud_rate"                : BAUD
        },
        "tx_rx_los_distance_m"         : 2.00,
        "tx_rx_axis"                   : "parallel to north-south axis"
    },

    # ── Firmware / Wi-Fi parameters ────────────────────────────────
    "wifi": {
        "ssid"              : "CSI_AP",              # <-- confirm
        "channel"           : 11,                     # <-- confirm
        "bandwidth_mhz"     : 40,
        "icmp_rate_hz"      : 30,                    # ping rate at TX
        "protocol"          : "ESP-NOW HT40"
    },

    # ── Collection protocol ────────────────────────────────────────
    "protocol": {
        "stabilization_s"   : 60,    # seconds of link stabilization before condition starts
        "active_window_s"   : 600,   # labeled collection window (10 min)
        "buffer_end_s"      : 30,    # safety buffer before stopping recording
        "subject_position"  : {"x_from_west_m": 1.45, "y_from_north_m": 2.11},
        "subject_orientation": "facing north (towards RX node)"
    },

    # ── Label descriptions ─────────────────────────────────────────
    "labels_description": {
        "empty"             : "No human presence; operator outside the room with door closed.",
        "occupied_still"    : "Subject standing at center mark, motionless.",
        "occupied_moving"   : "Subject standing at center mark, slow movements within ±0.30 m."
    }
}

print("✅ Setup metadata loaded.")
print(json.dumps(SETUP_META, indent=2, ensure_ascii=False))


---
## Cell 3 — Session parameters
Fill in **before each individual session**.  
> `SESSION_ID` is sequential per day: `A`, `B`, `C` ...  
> `LABEL` must be exactly: `"empty"`, `"occupied_still"` or `"occupied_moving"`


In [ ]:
#  FILL IN MANUALLY BEFORE EACH SESSION

SESSION_ID   = "E"               # sequential per day: A, B, C ...
LABEL        = "occupied_moving"  # "empty" | "occupied_still" | "occupied_moving"
DURATION     = 690               # total recording duration in seconds (>= stab + active + buffer)
START_DELAY  = 15                # countdown in seconds before recording starts

STABILIZATION_S  = SETUP_META["protocol"]["stabilization_s"]   # 60
ACTIVE_WINDOW_S  = SETUP_META["protocol"]["active_window_s"]    # 600

# Environmental conditions at the time of the session
ENV_COND = {
    "temperature_C"          : None,   # <-- fill in (e.g. 24)
    "visible_wifi_networks"  : None,   # <-- fill in (e.g. 3)
    "avg_rssi_dbm"           : None,   # <-- fill in after session (e.g. -45)
    "observed_interferences" : None,   # <-- fill in (e.g. "none" or "microwave in adjacent room")
    "door_state"             : "closed",  # "closed" | "open"
    "window_state"           : "closed",  # "closed" | "open"
    "free_notes"             : ""      # free-text field
}

# Basic validations
_valid_labels = {"empty", "occupied_still", "occupied_moving"}
assert LABEL in _valid_labels, f"Invalid LABEL. Use one of: {_valid_labels}"
assert SESSION_ID.isalpha() and len(SESSION_ID) <= 3, "SESSION_ID must be letter(s): A, B, C ..."

_dt_session   = datetime.now()
_base_name    = _build_filename(SESSION_ID, LABEL, _dt_session)
#CSV_FILE      = f"{_base_name}.csv"
#JSON_FILE     = f"{_base_name}_meta.json"
CSV_FILE      = str(DATA_DIR / f"{_base_name}.csv")
JSON_FILE     = str(DATA_DIR / f"{_base_name}_meta.json")

print(f"Session ID   : {SESSION_ID}")
print(f"Label        : {LABEL}")
print(f"Duration     : {DURATION}s")
print(f"CSV file     : {CSV_FILE}")
print(f"JSON file    : {JSON_FILE}")
print(f"\nDouble-check the label and session ID before running Cell 4.")


---
## Cell 4 — Data collection
Starts the serial recording.  
The script waits `START_DELAY` seconds before opening the port — use that time to leave the room (empty session) or take your position (occupied session).

> **Write down the exact time you enter/leave the room.** You will need those timestamps in Cell 5.


In [ ]:
from datetime import timedelta

# Initialize runtime metadata structure

_meta_runtime = {
    "t0_recording_start" : None,
    "t3_recording_stop"  : None,
    "total_samples"      : 0,
    "status"             : "PENDING",  # VALID | INVALID
    "invalidation_reason": ""
}

print(f"Starting in {START_DELAY}s — leave the room / take your position now...")
time.sleep(START_DELAY)

_t_start = datetime.now()
_meta_runtime["t0_recording_start"] = _ts_iso()
print(f"\nRECORDING -> {CSV_FILE}")
print(f"   Start : {_meta_runtime['t0_recording_start']}")
print(f"   Planned duration: {DURATION}s\n")

samples = 0

with serial.Serial(PORT, BAUD, timeout=1) as ser, \
     open(CSV_FILE, "w") as f:

    f.write(HEADER)
    t0 = time.time()

    while time.time() - t0 < DURATION:
        line = ser.readline().decode("utf-8", errors="ignore").strip()
        if line.startswith("CSI_DATA"):
            ts = _ts_host()
            f.write(f"{ts},{line}\n")
            samples += 1
            print(f"\r{samples} samples | {time.time()-t0:.1f}s", end="", flush=True)

_meta_runtime["t3_recording_stop"] = _ts_iso()
_meta_runtime["total_samples"]     = samples

print(f"\n\nRecording finished: {samples} samples -> '{CSV_FILE}'")
print(f"   Stop : {_meta_runtime['t3_recording_stop']}")
print("\nProceed to Cell 5 to log the ground-truth timestamps and export the JSON.")

_t1_auto = _t_start + timedelta(seconds=STABILIZATION_S)
_t2_auto = _t1_auto + timedelta(seconds=ACTIVE_WINDOW_S)

print(f"\nt1 (automatic): {_t1_auto.strftime('%H:%M:%S')}")
print(f"t2 (automatic): {_t2_auto.strftime('%H:%M:%S')}")
print("   Copy these values into T1 and T2 in Cell 5.")


---
### Calculating `avg_rssi_dbm`



In [ ]:
import pandas as pd

df = pd.read_csv(CSV_FILE)
rssi_mean = round(df["rssi"].mean(), 2)
print(f"avg_rssi_dbm → {rssi_mean}")

---
## Cell 5 — Wrap-up, ground truth & JSON export

Fill in `T1` and `T2` with the times you noted during the session:  
- **`T1`** → moment the condition started (left the room for *empty*; entered and took position for *occupied*)
- **`T2`** → moment the condition ended

Accepted format: `"HH:MM:SS"` — the date is automatically inferred from `t0`.

In [ ]:
#  FILL IN WITH THE TIMESTAMPS NOTED DURING THE SESSION

T1 = None   # leave None to use the automatic value calculated in Cell 4
T2 = None   # leave None to use the automatic value calculated in Cell 4

if T1 is None:
    T1 = _t1_auto.strftime("%H:%M:%S")
    print(f"T1 (automatic): {T1}")
if T2 is None:
    T2 = _t2_auto.strftime("%H:%M:%S")
    print(f"T2 (automatic): {T2}")

SESSION_STATUS       = "VALID"    # "VALID" | "INVALID"
INVALIDATION_REASON  = ""         # fill in if INVALID

# Complete any environmental fields left as None in Cell 3
ENV_COND["avg_rssi_dbm"]           = rssi_mean   # <-- fill in now (e.g. -45)
ENV_COND["temperature_C"]          = None   # <-- fill in if not done before
ENV_COND["visible_wifi_networks"]  = 8   # <-- fill in if not done before

# Automatic JSON construction
_date_ref = _t_start.date().isoformat()

def _to_iso(date_str: str, time_str: str) -> str:
    dt = datetime.fromisoformat(f"{date_str}T{time_str}")
    return dt.astimezone().isoformat(timespec='seconds')

_meta_runtime["t1_condition_start"] = _to_iso(_date_ref, T1)
_meta_runtime["t2_condition_end"]   = _to_iso(_date_ref, T2)
_meta_runtime["status"]             = SESSION_STATUS
_meta_runtime["invalidation_reason"]= INVALIDATION_REASON

metadata = {
    "schema_version" : "2.0",
    "session": {
        "id"           : SESSION_ID,
        "label"        : LABEL,
        "files": {
            "csv"      : CSV_FILE,
            "json"     : JSON_FILE
        },
        "planned_duration_s": DURATION
    },
    "timing"         : _meta_runtime,
    "environment"    : ENV_COND,
    "setup"          : SETUP_META
}

with open(JSON_FILE, "w", encoding="utf-8") as jf:
    json.dump(metadata, jf, indent=2, ensure_ascii=False)

print(f"JSON exported: {JSON_FILE}")

if SESSION_STATUS == "INVALID":
    print(f"\nSession marked as INVALID: {INVALIDATION_REASON}")
    print("   Files were saved but must NOT be included in the dataset.")
else:
    active_window = (
        datetime.fromisoformat(_meta_runtime['t2_condition_end']) -
        datetime.fromisoformat(_meta_runtime['t1_condition_start'])
    ).seconds
    print(f"\nSession summary:")
    print(f"   Status           : {SESSION_STATUS}")
    print(f"   Label            : {LABEL}")
    print(f"   Total samples    : {_meta_runtime['total_samples']}")
    print(f"   Active window    : {active_window}s  (t1 -> t2)")
    print(f"   t0 (rec. start)  : {_meta_runtime['t0_recording_start']}")
    print(f"   t1 (cond. start) : {_meta_runtime['t1_condition_start']}")
    print(f"   t2 (cond. end)   : {_meta_runtime['t2_condition_end']}")
    print(f"   t3 (rec. stop)   : {_meta_runtime['t3_recording_stop']}")
    print(f"\nFiles ready for the dataset:")
    print(f"   -> {CSV_FILE}")
    print(f"   -> {JSON_FILE}")


---
## Cell 6 — Inspect the generated JSON *(optional)*
Useful for verifying the metadata file contents before archiving.

In [ ]:
with open(JSON_FILE, encoding="utf-8") as jf:
    _contents = json.load(jf)

print(json.dumps(_contents, indent=2, ensure_ascii=False))


---
## Cell 7 — List sessions of the day *(optional)*
Lists all `csv + json` pairs found in the current working directory.

In [ ]:
from pathlib import Path

csvs  = sorted(DATA_DIR.glob("session_*.csv"))
jsons = {p.stem for p in DATA_DIR.glob("session_*_meta.json")}

print(f"{'CSV file':<55} {'JSON?':>6} {'Size':>10}")
print("-" * 75)
for csv_path in csvs:
    json_key = csv_path.stem + "_meta"
    has_json = "[OK]" if json_key in jsons else "[MISSING]"
    size     = f"{csv_path.stat().st_size / 1024:.1f} KB"
    print(f"{csv_path.name:<55} {has_json:>6} {size:>10}")

if not csvs:
    print("No session files found in the current directory.")
